# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: "Visible & Under-Clicked" — flag pages that already earn
meaningful search visibility (real impressions) but under-perform
their position tier's expected CTR. These are pages that don't need
new content or a ranking push — they need a metadata/title/snippet
fix, because the visibility is already there and the click isn't
converting the way pages at that position tier normally do.

Score: gsc_impressions * max(0, tier_avg_ctr - page_ctr)
  — rewards pages with BOTH real volume AND a real CTR shortfall
  relative to peers at the same position; a page with no shortfall
  or no volume scores near zero either way.

Reason code (ONE): ctr_underperforms_position_tier
Action label (ONE): review_metadata_snippet

Signals this leans on, both tied to real FlyRank flags named in the
session:
1. CTR-vs-position (behind the CTR-fix logic)
2. Volume/impressions (behind the quick-win flag)

In [2]:
%pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

# Pull month-level aggregates per page, real GSC columns only
base = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
base["ctr"] = base["clicks"] / base["impressions"]

# Position tiers, same buckets the lane guide uses
def tier(p):
    if p <= 3: return "top_3"
    elif p <= 10: return "page_1"
    elif p <= 20: return "striking"
    elif p <= 50: return "page_3_5"
    else: return "deep"
base["position_tier"] = base["avg_position"].apply(tier)

# SIGNAL CHECK 1 — CTR vs position tier (behind CTR-fix logic)
signal1 = base.groupby("position_tier").agg(
    mean_ctr=("ctr", "mean"), n=("ctr", "count")
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Signal 1 — CTR by position tier:")
print(signal1)
print("\nVerdict: [fill CONFIRMED if CTR clearly drops as tier worsens top_3 -> deep, else OPPOSITE/MIXED/FALSE]")

# SIGNAL CHECK 2 — volume vs position (behind quick-win logic)
base["volume_tier"] = pd.qcut(base["impressions"], 4, labels=["low", "medium", "high", "very_high"])
signal2 = base.groupby("volume_tier").agg(
    mean_position=("avg_position", "mean"), n=("avg_position", "count")
)
print("\nSignal 2 — avg_position by volume tier:")
print(signal2)
print("\nVerdict: [fill — does higher volume alone predict better position? "
      "If NOT clearly, that's an honest MIXED/FALSE — volume alone isn't proof of ranking quality]")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 — CTR by position tier:
               mean_ctr      n
position_tier                 
top_3          0.012399  17578
page_1         0.004926  81988
striking       0.003211  32203
page_3_5       0.002287  33288
deep           0.000903  11681

Verdict: [fill CONFIRMED if CTR clearly drops as tier worsens top_3 -> deep, else OPPOSITE/MIXED/FALSE]

Signal 2 — avg_position by volume tier:
             mean_position      n
volume_tier                      
low              15.378710  44983
medium           22.032410  43409
high             15.692132  44186
very_high        11.008204  44160

Verdict: [fill — does higher volume alone predict better position? If NOT clearly, that's an honest MIXED/FALSE — volume alone isn't proof of ranking quality]


/tmp/ipykernel_456/3936425023.py:43: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = base.groupby("volume_tier").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = impressions * max(0, tier_avg_ctr - page_ctr). Reason code
and action label are constant (this is one rule, not a multi-branch
system) — every flagged page gets the same explanation because
they're all being flagged for the same reason.

In [3]:
import os

tier_avg_ctr = base.groupby("position_tier")["ctr"].transform("mean")
base["ctr_gap"] = (tier_avg_ctr - base["ctr"]).clip(lower=0)
base["score"] = base["impressions"] * base["ctr_gap"]

base["reason_code"] = "ctr_underperforms_position_tier"
base["action"] = "review_metadata_snippet"

queue = base.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue[["content_hash_id", "position_tier", "impressions", "clicks", "ctr",
       "avg_position", "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print(queue.head(20)[["content_hash_id", "position_tier", "impressions", "ctr", "avg_position", "score"]])

Wrote 176738 rows to work/outputs/baseline_action_score.csv
             content_hash_id position_tier  impressions       ctr  \
0   content_8d7d99f109e19aa2         top_3     203497.0  0.001420   
1   content_0e03de7680314cd5         top_3     221310.0  0.003253   
2   content_eadb33b5df496f4a         top_3     617124.0  0.009185   
3   content_4ffe18112a5642e3         top_3     186983.0  0.003134   
4   content_ec2e0346994fb5a5         top_3     245276.0  0.006034   
5   content_545bb6cc7081ded3         top_3     122905.0  0.002335   
6   content_44f34c0a90047651        page_1     212404.0  0.000113   
7   content_9ef3d7516483e665         top_3      89229.0  0.001031   
8   content_306bc78dff1eb683         top_3      80821.0  0.000433   
9   content_987d251ee617d9c6         top_3     152806.0  0.006152   
10  content_c46df0fa61530d86         top_3      70398.0  0.000597   
11  content_80eb6221de550658         top_3      79766.0  0.002194   
12  content_e0ca055423cbe896         top_3 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = queue.head(20)[["content_hash_id", "position_tier", "impressions", "ctr", "avg_position", "score"]]
print(top20.to_string())


             content_hash_id position_tier  impressions       ctr  avg_position        score
0   content_8d7d99f109e19aa2         top_3     203497.0  0.001420      2.563756  2234.248399
1   content_0e03de7680314cd5         top_3     221310.0  0.003253      2.675217  2024.119585
2   content_eadb33b5df496f4a         top_3     617124.0  0.009185      2.383011  1983.990668
3   content_4ffe18112a5642e3         top_3     186983.0  0.003134      2.331060  1732.484083
4   content_ec2e0346994fb5a5         top_3     245276.0  0.006034      2.854514  1561.284512
5   content_545bb6cc7081ded3         top_3     122905.0  0.002335      2.615390  1236.952906
6   content_44f34c0a90047651        page_1     212404.0  0.000113      7.346909  1022.313231
7   content_9ef3d7516483e665         top_3      89229.0  0.001031      2.481596  1014.389438
8   content_306bc78dff1eb683         top_3      80821.0  0.000433      1.488604   967.134964
9   content_987d251ee617d9c6         top_3     152806.0  0.006152     

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weakest picks: rows near the bottom of the top 20 where impressions
are just barely above the volume floor — the score there is driven
more by a slightly-larger-than-average CTR gap on modest volume than
by genuine, large missed opportunity. Worth a lower confidence note
on those specifically.

In [5]:
used_cols = ["content_hash_id", "impressions", "clicks", "ctr", "avg_position", "position_tier"]
forbidden = ["health_score", "priority_score", "action_type", "refresh_tier", "trend_pct", "trend_direction"]
leaked = [c for c in used_cols if c in forbidden]
print("Feature columns actually used:", used_cols)
print("Any forbidden/product-flag or label-derived columns present:", leaked if leaked else "none — clean")
print("Time window used: single month (2026-03), no future window referenced.")


Feature columns actually used: ['content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'position_tier']
Any forbidden/product-flag or label-derived columns present: none — clean
Time window used: single month (2026-03), no future window referenced.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.